In [1]:
import pandas as pd
from datetime import datetime

# 1. CARGA DE DATOS
print("--- Cargando transacciones ---")
# Leemos el archivo generado en el paso anterior
df = pd.read_csv('ecommerce_transactions.csv')

# CONVERSIÓN CRÍTICA: Convertir texto a objetos de fecha reales
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. PUNTO DE REFERENCIA (SNAPSHOT)
# Para calcular "hace cuántos días compró", necesitamos definir el "hoy" del análisis.
# Usamos el día siguiente a la última venta registrada en el dataset.
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f"Fecha de corte del análisis: {snapshot_date.date()}")

# 3. AGREGACIÓN (La magia del RFM)
# Agrupamos por ID de Cliente y calculamos las 3 métricas clave
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: Días desde la última compra
    'InvoiceNo': 'count',                                    # Frequency: Cantidad de compras
    'TotalAmount': 'sum'                                     # Monetary: Dinero total gastado
})

# 4. LIMPIEZA Y RENOMBRADO
# Renombramos las columnas para que sean técnicas (R, F, M)
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

# Verificación de Negocio
print("\n--- Muestra de la Matriz RFM ---")
print(rfm.head())

print("\n--- Estadísticas de tus Clientes ---")
print(rfm.describe())

# 5. GUARDADO
# Guardamos este dataset limpio para que la IA lo use en el siguiente paso
rfm.to_csv('rfm_data.csv')
print("\n✅ Archivo 'rfm_data.csv' generado exitosamente.")

--- Cargando transacciones ---
Fecha de corte del análisis: 2025-12-29

--- Muestra de la Matriz RFM ---
            Recency  Frequency  Monetary
CustomerID                              
1000            255          2    260.65
1001             61          8     96.62
1002             16         14    449.81
1003              4         18    424.54
1004              5         62   2481.30

--- Estadísticas de tus Clientes ---
          Recency   Frequency     Monetary
count  650.000000  650.000000   650.000000
mean    60.066154   15.384615   474.860923
std     75.914255   19.514202   649.683977
min      1.000000    1.000000     1.700000
25%      9.000000    3.000000    69.400000
50%     28.000000    8.000000   217.575000
75%     77.000000   19.000000   614.640000
max    363.000000  136.000000  4774.020000

✅ Archivo 'rfm_data.csv' generado exitosamente.
